# ⚡ XGBoost Model for Credit Card Fraud (Kaggle PGS S6E11)

Welcome to the **XGBoost Benchmark Notebook** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🎯 Objectives
1. Implement a GPU-accelerated **XGBoost 5-Fold Stratified Cross-Validation** pipeline (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`).
2. Utilize shallow depth (`max_depth = 3`) to match our shallow-tree finding from LightGBM.
3. Leverage native categorical support (`enable_categorical = True`) with histogram tree method (`tree_method = 'hist'`).
4. Generate out-of-fold validation probabilities and benchmark predictions for ensembling.


In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"XGBoost Version: {xgb.__version__}")


## 1. Paths & Configuration

In [ ]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Model Hyperparameters
    MAX_DEPTH = 3
    LEARNING_RATE = 0.05
    N_ESTIMATORS = 1500
    EARLY_STOPPING_ROUNDS = 50
    SUBSAMPLE = 0.8
    COLSAMPLE_BYTREE = 0.8
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")


## 2. Data Ingestion & Categorical Formatting

In [ ]:
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# Convert categorical features to category dtype for XGBoost enable_categorical=True
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

# Convert boolean flags to integers
for b in bool_cols:
    train_df[b] = train_df[b].astype(int)
    test_df[b] = test_df[b].astype(int)

features = [c for c in train_df.columns if c not in ['id', 'is_fraud']]
print(f"Total Features: {len(features)}")
print(f"Categoricals: {cat_cols}")


## 3. 5-Fold Stratified Cross-Validation (GPU Accelerated)

In [ ]:
X = train_df[features]
y = train_df['is_fraud'].astype(int)
X_test = test_df[features]

skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_preds = np.zeros(len(X), dtype=np.float64)
test_preds = np.zeros(len(X_test), dtype=np.float64)
fold_scores = []

# Detect GPU capability
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")

print("=" * 65)
print(f"Training XGBoost (max_depth={Config.MAX_DEPTH}, device={device})...")
print("=" * 65)

start_all = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    t0 = time.time()
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        max_depth=Config.MAX_DEPTH,
        learning_rate=Config.LEARNING_RATE,
        n_estimators=Config.N_ESTIMATORS,
        subsample=Config.SUBSAMPLE,
        colsample_bytree=Config.COLSAMPLE_BYTREE,
        tree_method='hist',
        device=device,
        enable_categorical=True,
        early_stopping_rounds=Config.EARLY_STOPPING_ROUNDS,
        random_state=Config.RANDOM_STATE + fold
    )
    
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        verbose=False
    )
    
    val_probs = model.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = val_probs
    test_preds += model.predict_proba(X_test)[:, 1] / Config.N_SPLITS
    
    fold_auc = roc_auc_score(y_val_fold, val_probs)
    fold_pr_auc = average_precision_score(y_val_fold, val_probs)
    fold_scores.append((fold_auc, fold_pr_auc))
    
    best_iter = getattr(model, 'best_iteration', -1)
    print(f"Fold {fold} | Best Iter: {best_iter:4d} | ROC-AUC: {fold_auc:.5f} | PR-AUC: {fold_pr_auc:.5f} ({time.time()-t0:.1f}s)")

total_cv_time = time.time() - start_all
print("=" * 65)
print(f"All 5 Folds Completed in {total_cv_time:.1f}s!")


## 4. Performance Evaluation & Diagnostics

In [ ]:
overall_auc = roc_auc_score(y, oof_preds)
overall_pr_auc = average_precision_score(y, oof_preds)

print("-" * 55)
print("XGBoost 5-Fold Cross-Validation Summary:")
print(f"Mean Fold ROC-AUC:   {np.mean([s[0] for s in fold_scores]):.5f} ± {np.std([s[0] for s in fold_scores]):.5f}")
print(f"Overall OOF ROC-AUC: {overall_auc:.5f}")
print(f"Overall OOF PR-AUC:  {overall_pr_auc:.5f}")
print("-" * 55)

# Save OOF predictions for ensembling
np.save(Config.OUTPUT_DIR / 'oof_xgboost.npy', oof_preds)
print("Saved OOF predictions for ensembling to oof_xgboost.npy")

# Plot Diagnostic Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='#e65100', lw=2, label=f'OOF ROC (AUC = {overall_auc:.5f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (5-Fold OOF XGBoost)')
axes[0].legend(loc='lower right')

precision, recall, _ = precision_recall_curve(y, oof_preds)
axes[1].plot(recall, precision, color='#0277bd', lw=2, label=f'OOF PR (AUC = {overall_pr_auc:.5f})')
axes[1].axhline(y=y.mean(), color='gray', linestyle='--', label=f'Base Rate ({y.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (5-Fold OOF XGBoost)')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('xgboost_diagnostics.png', dpi=150)
plt.show()


## 5. Submission File Generation & Verification

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': test_preds
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'xgboost_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"Submission successfully generated and verified at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))
